In [12]:
import json
import os
import shutil
from pathlib import Path

import yaml
from rdflib import Dataset, Graph
from pyshacl import validate

SRCDIR = Path(".")
BUILDDIR = SRCDIR / ".." / "build"
DATADIR = Path("data")

def yaml2json(y):
    os.makedirs(BUILDDIR / y.parent, exist_ok=True)
    with y.open("rb") as f:
        data = yaml.safe_load(f)
    with open(BUILDDIR / y.with_suffix(".jsonld"), "w") as g:
        json.dump(data, g, default=str)

shutil.rmtree(BUILDDIR, ignore_errors=True)

yaml2json(SRCDIR / "remote" / "base-voltage.shacl.yamlld")
yaml2json(SRCDIR / "remote" / "nbnl-prof.context.yamlld")
yaml2json(SRCDIR / "nbnl-equipment.rdfs.yamlld")
yaml2json(SRCDIR / "nbnl-equipment.prof.yamlld")
shutil.copy(SRCDIR / "remote" / "cim17.rdfs.ttl", BUILDDIR / "remote")

data_dataset = Dataset()
data_dataset.parse(
    DATADIR / "AS_EQ_OP_Westland_FGU" / "equipment_four_stations_EQ.jsonld",
    format="json-ld",
)

shapes_graph = Graph()
shapes_graph.parse(BUILDDIR / "nbnl-equipment.prof.jsonld", format="json-ld")

cim_ontology_graph = Graph()
cim_ontology_graph.parse(
    BUILDDIR / "remote" / "cim17.rdfs.ttl", format="turtle"
)  # TODO: Should be the profile vocab which imports this.

conforms, results_graph, results_text = validate(
    data_graph=data_dataset,
    shacl_graph=shapes_graph,
    ont_graph=cim_ontology_graph,  # TODO: This is very specific to this validator. Can we do this differently?
    meta_shacl=True,
    inference="rdfs",
    do_owl_imports=True,
)

print(results_text)

ParseException: Expected SelectQuery, found '1'  (at char 218), (line:6, col:22)